<H3>This assignment demonstrates the implementation and evaluation of an autoencoder using the Fashion-MNIST dataset. The objective is to build an encoder and decoder network, train them jointly to reconstruct input images, and evaluate the model’s ability to reproduce original images by visually comparing reconstructed outputs with their corresponding inputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, datasets

In [ ]:
(x_train, _), (x_test, _) = datasets.fashion_mnist.load_data()

In [ ]:
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test  = (x_test.astype("float32")  / 255.0)[..., np.newaxis]

In [ ]:
LATENT_DIM = 64  # you can adjust; keep fixed for your submission

encoder_input = layers.Input(shape=(28, 28, 1), name="encoder_input")
x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(encoder_input)
x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
x = layers.Flatten()(x)
latent = layers.Dense(LATENT_DIM, name="latent_vector")(x)

encoder = models.Model(encoder_input, latent, name="encoder")
encoder.summary()

In [ ]:
decoder_input = layers.Input(shape=(LATENT_DIM,), name="decoder_input")

In [ ]:
x = layers.Dense(7 * 7 * 64, activation="relu")(decoder_input)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)  # 7->14
x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)  # 14->28
decoder_output = layers.Conv2D(1, 3, padding="same", activation="sigmoid", name="decoder_output")(x)

decoder = models.Model(decoder_input, decoder_output, name="decoder")
decoder.summary()

In [ ]:
autoencoder_input = encoder_input
encoded = encoder(autoencoder_input)
reconstructed = decoder(encoded)

autoencoder = models.Model(autoencoder_input, reconstructed, name="autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()


In [ ]:
history = autoencoder.fit(
    x_train, x_train,
    epochs=10,
    batch_size=128,
    shuffle=True,
    validation_split=0.1,
    verbose=1
)

In [ ]:
chosen_indices = [0, 1, 2, 3, 4]

In [ ]:
x_sample = x_test[chosen_indices]
x_recon = autoencoder.predict(x_sample, verbose=0)

In [ ]:
n = len(chosen_indices)
plt.figure(figsize=(12, 4))

In [ ]:
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    ax.imshow(x_sample[i].squeeze(), cmap="gray")
    ax.set_title(f"Original\nidx={chosen_indices[i]}")
    ax.axis("off")
    ax = plt.subplot(2, n, i + 1 + n)
    ax.imshow(x_recon[i].squeeze(), cmap="gray")
    ax.set_title("Reconstructed")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Autoencoder Reconstruction Loss")
plt.legend()
plt.show()